In [119]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import yfinance as yf
import ta
import math
from fredapi import Fred
import logging
import ecbdata

# List Of Tradeable Pairs And Indicators

In [120]:
# Initialize MetaTrader 5 connection
mt5.initialize()

# Updated list of currency pairs
pairs = [
    "EURUSD",  # Euro / US Dollar
    "EURCHF",  # Euro / Swiss Franc
    "EURJPY",  # Euro / Japanese Yen
    "USDCHF",  # US Dollar / Swiss Franc
    "CHFJPY",  # Swiss Franc / Japanese Yen
    
    "USDJPY"   # US Dollar / Japanese Yen
]

currencies = [
   "DX-Y.NYB", # Dollar Currency Index
    "^XDE",    # Euro Currency Index
    "^XDS",    # Chf Currency Index
    "^XDN"     # Yen Currency Index
]

# Function to get the latest Ask and Bid prices for a given pair
def get_latest_prices(symbol):
    # Get the latest tick data for the symbol
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        print(f"Failed to get latest tick data for {symbol}")
        return None, None
    return tick.ask, tick.bid

# Function to get historical data for a given pair
def get_historical_data(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    # Fetch historical data
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, n_bars)
    if rates is None or len(rates) == 0:
        print(f"Failed to get historical data for {symbol}")
        return None
    data = pd.DataFrame(rates)
    data['time'] = pd.to_datetime(data['time'], unit='s')
    return data

# Function to calculate EMA, RSI, and ATR for a given pair
def calculate_indicators(symbol):
    # Get historical data for the pair
    data = get_historical_data(symbol)
    if data is None:
        return None, None, None, None

    # Calculate EMA 64
    data['EMA_64'] = ta.trend.ema_indicator(data['close'], window=64)

    # Calculate RSI 16
    data['RSI_16'] = ta.momentum.rsi(data['close'], window=16)

    # Calculate ATR 16
    data['ATR_16'] = ta.volatility.average_true_range(data['high'], data['low'], data['close'], window=16)

    # Get the latest values of the indicators
    latest_price = data['close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    latest_atr = data['ATR_16'].iloc[-1]

    return latest_price, latest_ema, latest_rsi, latest_atr

# Macroeconomic Data

In [121]:
import pandas as pd
from fredapi import Fred
import logging
from ecbdata import ecbdata

# Initialize FRED with your API key
fred = Fred(api_key='e886df7269c2c4e6209754d4ea0371d5')  # Replace with your actual API key

# Define the countries and their respective indicators
countries = {
    'USA': {'gdp': 'GDPC1', 'unemp': 'UNRATE', 'interest': 'FEDFUNDS', 'inflation': 'CPIAUCSL'},
    'Europe': {'gdp': 'CLVMEURSCAB1GQEA19', 'unemp': 'LRUNTTTTQZA156S', 'interest': 'IR3TIB01EZQ156N', 'inflation': 'CPHPTT01EZQ659N'},
    'Switzerland': {'gdp': 'CLVMNACSAB1GQCH', 'unemp': 'LRUNTTTTCHQ156S', 'interest': 'IR3TIB01CHQ156N', 'inflation': 'CHECPIALLMINMEI'},
    'Japan': {'gdp': 'JPNRGDPEXP', 'unemp': 'LRUNTTTTJPQ156S', 'interest': 'IR3TIB01JPQ156N', 'inflation': 'JPNCPIALLMINMEI'}
}


# Initialize empty dictionaries for storing the economic data
growth_rates = {}
unemp_rates = {}
interest_rates = {}
inflation_rates = {}

def get_10_years_data(symbol, country, indicator):
    """Fetch last 10 years of data for a given symbol with fallback"""
    try:
        data = fred.get_series(symbol)
        data = data[data.index >= pd.Timestamp.now() - pd.DateOffset(years=10)]
        if data.empty:
            raise ValueError(f"No data returned for {symbol}")
        logging.info(f"Successfully fetched {indicator} data for {country} using {symbol} ({len(data)} points)")
        return data
    except Exception as e:
        # logging.warning(f"Error fetching {indicator} data for {country} with {symbol}: {e}")
        if country in fallbacks and indicator in fallbacks[country]:
            fallback_symbol = fallbacks[country][indicator]
            logging.info(f"Attempting fallback {fallback_symbol} for {country} {indicator}")
            try:
                data = fred.get_series(fallback_symbol)
                data = data[data.index >= pd.Timestamp.now() - pd.DateOffset(years=10)]
                if data.empty:
                    raise ValueError(f"No data returned for fallback {fallback_symbol}")
                logging.info(f"Successfully used fallback {fallback_symbol} for {country} {indicator} ({len(data)} points)")
                return data
            except Exception as e:
                logging.error(f"Fallback {fallback_symbol} failed for {country} {indicator}: {e}")
        return None

def calculate_growth_rate(gdp_series):
    """Calculate annualized quarterly GDP growth rate"""
    if gdp_series is None or len(gdp_series) < 2:
        return None
    gdp_now = gdp_series.iloc[-1]
    gdp_previous = gdp_series.iloc[-2]
    return ((gdp_now - gdp_previous) / gdp_previous) * 100 * 4

def get_latest_value(series):
    """Get the most recent value from a series"""
    if series is None or len(series) < 1:
        return None
    return series.iloc[-1]

def calculate_inflation_rate(cpi_series):
    """Calculate year-over-year inflation rate"""
    if cpi_series is None or len(cpi_series) < 13:  # Need 12 months + 1 for monthly data
        logging.warning(f"Insufficient data for inflation calculation: {len(cpi_series)} points")
        return None
    cpi_now = cpi_series.iloc[-1]
    cpi_year_ago = cpi_series.iloc[-13]  # Assumes monthly data
    return ((cpi_now - cpi_year_ago) / cpi_year_ago) * 100

# Update the economic data and store it in a DataFrame
def update_economic_data():
    global growth_rates, unemp_rates, interest_rates, inflation_rates
    for country, symbols in countries.items():
        # GDP Growth
        gdp_data = get_10_years_data(symbols['gdp'], country, 'gdp')
        if gdp_data is not None:
            growth_rates[country] = calculate_growth_rate(gdp_data)
        
        # Unemployment
        unemp_data = get_10_years_data(symbols['unemp'], country, 'unemployment')
        if unemp_data is not None:
            unemp_rates[country] = get_latest_value(unemp_data)
        
        # Interest Rates
        interest_data = get_10_years_data(symbols['interest'], country, 'interest')
        if interest_data is not None:
            interest_rates[country] = get_latest_value(interest_data)
        
        # Inflation
        inflation_data = get_10_years_data(symbols['inflation'], country, 'inflation')
        if inflation_data is not None:
            if country == 'Europe' and symbols['inflation'] == 'CPHPTT01EZQ659N':
                inflation_rates[country] = get_eur_inflation()
            else:
                inflation_rates[country] = calculate_inflation_rate(inflation_data)

def get_eur_inflation():
    data = ecbdata.get_series('ICP.M.U2.N.000000.4.ANR', 
                        start='2024-01')
    latest = data['OBS_VALUE'].iloc[-1]
    return latest

def get_eur_unemployment():
    data = ecbdata.get_series('LFSI.M.U2.N.UNEHRT.TOTAL0.15_74.T', 
                        start='2024-01')
    latest = data['OBS_VALUE'].iloc[-1]
    return latest

# Create a DataFrame to hold the economic data
def create_economic_dataframe():
    update_economic_data()
    
    # Combine the data into a DataFrame
    data = {
        'GDP Growth': growth_rates,
        'Unemployment': unemp_rates,
        'Interest Rate': interest_rates,
        'Inflation Rate': inflation_rates
    }
    
    df = pd.DataFrame(data)
    df.loc['Japan', "Inflation Rate"] = 4
    df.loc['Europe', 'Unemployment'] = get_eur_unemployment() 
    return df
df = create_economic_dataframe()

df


,GDP Growth,Unemployment,Interest Rate,Inflation Rate
USA,2.232707,4.000000,4.330000,2.999413
Europe,0.203240,6.268223,2.996487,2.500000
Switzerland,1.536775,4.497941,0.767250,0.401528
Japan,1.233826,2.466667,0.334667,4.000000


# Currency Index Data

# Pip Value

In [122]:
def get_pip_value(symbol):
    dec = 0.0001
    if "JPY" in symbol:
        dec = 0.01
    latest_price, latest_ema, latest_rsi, latest_atr = calculate_indicators(symbol)
    pip_value = (dec * 100000) / latest_price
    return pip_value

# Position Size

In [123]:
def get_position_size(pair, stop_loss):
    account_info = mt5.account_info()
    balance = account_info.balance
    risk_amount = 0.005 * balance
    size = risk_amount / ( stop_loss * get_pip_value(pair))
    size = round(size, 2)
    return size

# Bias

In [124]:
def compare_economies(country1, country2):
    """
    Compare the economic data between two countries.
    Returns a 'buy' signal for country1 or 'sell' for country2 based on macroeconomic performance.
    If 2/4 factors are in favor of each country, returns 'neutral'.
    """
    buy_factors = 0
    sell_factors = 0
    
    # Compare GDP Growth Rates
    if growth_rates.get(country1, 0) > growth_rates.get(country2, 0):
        buy_factors += 1
    elif growth_rates.get(country1, 0) < growth_rates.get(country2, 0):
        sell_factors += 1
    
    # Compare Unemployment Rates (lower is better)
    if unemp_rates.get(country1, float('inf')) < unemp_rates.get(country2, float('inf')):
        buy_factors += 1
    elif unemp_rates.get(country1, float('inf')) > unemp_rates.get(country2, float('inf')):
        sell_factors += 1
    
    # Compare Interest Rates (higher is generally better)
    if interest_rates.get(country1, 0) > interest_rates.get(country2, 0):
        buy_factors += 1
    elif interest_rates.get(country1, 0) < interest_rates.get(country2, 0):
        sell_factors += 1
    
    # Compare Inflation Rates (lower is better)
    if inflation_rates.get(country1, float('inf')) < inflation_rates.get(country2, float('inf')):
        buy_factors += 1
    elif inflation_rates.get(country1, float('inf')) > inflation_rates.get(country2, float('inf')):
        sell_factors += 1
    
    # Determine the final bias based on the number of factors in each direction
    if buy_factors >= 3:
        return 'buy'
    elif sell_factors >= 3:
        return 'sell'
    else:
        return 'neutral'


def bias_for_pairs(pairs):
    bias_results = {}
    
    # Loop through each pair and get the macroeconomic comparison
    for i in pairs:
        if "EUR" in i and "USD" in i:
            bias = compare_economies("EU", "US")  # Compare Eurozone with USA
        elif "EUR" in i and "CHF" in i:
            bias = compare_economies("EU", "Switzerland")  # Compare Eurozone with Switzerland
        elif "EUR" in i and "JPY" in i:
            bias = compare_economies("EU", "Japan")  # Compare Eurozone with Japan
        elif "USD" in i and "CHF" in i:
            bias = compare_economies("US", "Switzerland")  # Compare USA with Switzerland
        elif "CHF" in i and "JPY" in i:
            bias = compare_economies("Switzerland", "Japan")  # Compare Switzerland with Japan
        elif "USD" in i and "JPY" in i:
            bias = compare_economies("US", "Japan")  # Compare USA with Japan
        
        # Update the dictionary with the bias result
        bias_results[i] = bias
    
    return bias_results


# Forex Pairs Data table

In [125]:
def get_data_table(pairs):
    dict = {}
    bias_results = bias_for_pairs(pairs)
    for i in pairs:
        ask, bid = get_latest_prices(i)
        latest_price, latest_ema, latest_rsi, latest_atr = calculate_indicators(i)
        dict.update({i: [ask,bid,latest_price, latest_ema, latest_rsi, latest_atr]})
    # Convert to DataFrame
    data = pd.DataFrame.from_dict(dict, orient='index', columns=['Ask', 'Bid', 'price', 'EMA', 'RSI', 'ATR'])
    
    data['Stop Loss']= data['ATR'] * 3
    data['Take Profit'] = data['Stop Loss'] * 2
    for i in pairs:
        dec = 10000
        if "JPY" in i:
            dec = 100
        stop_loss = round(data.loc[i, "Stop Loss"] * dec)
        take_profit = round(data.loc[i, "Take Profit"] * dec)
        data.loc[i,'Round Stop Loss'] = int(stop_loss)
        data.loc[i,'Round Take Profit'] = int(take_profit)
        data.loc[i,'Pip Value'] = get_pip_value(i)
        data.loc[i,'Position Size'] = get_position_size(i, stop_loss)
        data.loc[i, 'Bias'] = bias_results.get(i, 'neutral')
    return data
        
data = get_data_table(pairs)

data


,Ask,Bid,price,EMA,RSI,ATR,Stop Loss,Take Profit,Round Stop Loss,Round Take Profit,Pip Value,Position Size,Bias
EURUSD,1.04604,1.04601,1.04601,1.047003,45.138342,0.000617,0.001851,0.003701,19.0,37.0,9.560138,4.40,neutral
EURCHF,0.93877,0.93863,0.93863,0.940521,35.364016,0.000436,0.001307,0.002614,13.0,26.0,10.653825,5.78,sell
EURJPY,156.03300,156.01300,156.01300,156.807348,34.564665,0.145784,0.437353,0.874705,44.0,87.0,6.409722,2.84,sell
USDCHF,0.89745,0.89732,0.89732,0.898285,42.008476,0.000509,0.001526,0.003053,15.0,31.0,11.144296,4.79,sell
CHFJPY,166.23700,166.21300,166.21300,166.722124,39.412975,0.123785,0.371354,0.742709,37.0,74.0,6.016377,3.59,neutral
USDJPY,149.15200,149.14700,149.14700,149.764704,34.711731,0.125211,0.375633,0.751266,38.0,75.0,6.704795,3.14,sell
